# Sphota: GPS for meaning (Colab quick tour)

Any text, in any language → a short, stable, hierarchical code. This notebook runs on the free CPU runtime.

In [ ]:
!pip -q install 'sphota[encode,cli]'

In [ ]:
from sphota import encode, encode_batch, similarity, fingerprint, compare
c = encode('The CEO resigned')
c, c.label, [round(p, 2) for p in c.confidence]

## Same meaning, different languages

In [ ]:
for t in ['सीईओ ने इस्तीफा दे दिया', 'El director ejecutivo dimitió', 'Der Vorstandschef ist zurückgetreten', '首席执行官辞职了']:
    o = encode(t)
    print(o, 'shared levels:', c.common_prefix(o), '|', o.label)

## Precision like GPS accuracy, alternates near boundaries

In [ ]:
print(encode('I love you', min_confidence=0.5))
print(encode('I love you').alternates)

## Topic analytics on a CSV

In [ ]:
import pandas as pd
url = 'https://raw.githubusercontent.com/thiruvikramanpravartak/sphota/main/examples/data/headlines.csv'
df = pd.read_csv(url)
df['code'] = [str(x) for x in encode_batch(df.text.tolist())]
df['l1'] = df.code.str[:4]
df.groupby('l1').agg(n=('text','size'), example=('text','first')).sort_values('n', ascending=False).head(10)

In [ ]:
!sphota analyze headlines.csv -o report.html 2>/dev/null || (python -c "import pandas as pd; pd.read_csv('https://raw.githubusercontent.com/thiruvikramanpravartak/sphota/main/examples/data/headlines.csv').to_csv('headlines.csv', index=False)" && sphota analyze headlines.csv -o report.html)
from IPython.display import HTML
HTML(open('report.html').read())

## Never re-index: query an index built with another model

In [ ]:
from sphota.usecases import CodeIndex
idx = CodeIndex()
idx.add_texts(df.text.tolist(), model='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
idx.search('flooded streets after heavy rain', k=3)

## Document fingerprints

In [ ]:
a = 'The company reported strong earnings. Revenue grew. Heavy rain disrupted deliveries.'
b = 'The company reported strong earnings. The CEO resigned unexpectedly.'
compare(a, b), fingerprint(a).top(3)